<a href="https://colab.research.google.com/github/jayatigupta05/VEP/blob/main/VEP.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

#Step 0: Preparing Data

## Processing raw data

### Loading raw data

In [ ]:
import os
from google.colab import drive
drive.mount('/content/drive')

ValueError: mount failed

In [ ]:
import os
PROJECT_PATH = "/content/drive/MyDrive/variant_effect_prediction"
if not os.path.exists(PROJECT_PATH):
    os.makedirs(PROJECT_PATH)
    os.makedirs('data/raw', exist_ok=True)
    os.makedirs('data/processed', exist_ok=True)
    os.makedirs('src', exist_ok=True)
os.chdir(PROJECT_PATH)
print(f"Current working directory: {os.getcwd()}")

Current working directory: /content/drive/MyDrive/variant_effect_prediction


In [ ]:
import pandas as pd
RAW_FILE_PATH = '/content/drive/MyDrive/variant_effect_prediction/data/raw/variant_summary.txt'
# len(pd.read_csv(RAW_FILE_PATH, sep='\t', low_memory=False))

In [ ]:
with open(RAW_FILE_PATH, 'r') as f:
    for _ in range(10):
        print(f.readline().strip())

#AlleleID	Type	Name	GeneID	GeneSymbol	HGNC_ID	ClinicalSignificance	ClinSigSimple	LastEvaluated	RS# (dbSNP)	nsv/esv (dbVar)	RCVaccession	PhenotypeIDS	PhenotypeList	Origin	OriginSimple	Assembly	ChromosomeAccession	Chromosome	Start	Stop	ReferenceAllele	AlternateAllele	Cytogenetic	ReviewStatus	NumberSubmitters	Guidelines	TestedInGTR	OtherIDs	SubmitterCategories	VariationID	PositionVCF	ReferenceAlleleVCF	AlternateAlleleVCF	SomaticClinicalImpact	SomaticClinicalImpactLastEvaluated	ReviewStatusClinicalImpact	Oncogenicity	OncogenicityLastEvaluated	ReviewStatusOncogenicity	SCVsForAggregateGermlineClassification	SCVsForAggregateSomaticClinicalImpact	SCVsForAggregateOncogenicityClassification
15041	Indel	NM_014855.3(AP5Z1):c.80_83delinsTGCTGTAAACTGTAACTGTAAA (p.Arg27_Ile28delinsLeuLeuTer)	9907	AP5Z1	HGNC:22197	Pathogenic/Likely pathogenic	1	Dec 17, 2024	397704705	-	RCV000000012|RCV005255549|RCV004998069	MONDO:MONDO:0013342,MedGen:C3150901,OMIM:613647,Orphanet:306511||MedGen:C3661900	Hereditary spa

### Filtering and chunking


In [ ]:
import pandas as pd
import numpy as np

cols_to_keep = [
    'Type', 'GeneSymbol', 'ClinicalSignificance',
    'Assembly', 'Chromosome', 'Start', 'Stop',
    'ReferenceAlleleVCF', 'AlternateAlleleVCF',
    'NumberSubmitters', 'OriginSimple', 'ReviewStatus', 'SomaticClinicalImpact'
]

loc = "/content/drive/MyDrive/variant_effect_prediction/"
csv_reader = pd.read_csv(
    f'{loc}data/raw/variant_summary.txt',
    sep='\t',
    chunksize=100000,
    low_memory=False
)

print("Processing started...")

for index, chunk in enumerate(csv_reader):
    output_filename = f"{loc}data/processed/clinvar_clean_part_{index}.parquet"

    # if os.path.exists(output_filename):
    #     print(f"-> Part {index} already exists. Skipping to save time!")
    #     continue

    if index == 0 and '#AlleleID' in chunk.columns:
        chunk = chunk.rename(columns={'#AlleleID': 'AlleleID'})

    # Keep ONLY the modern human genome assembly (GRCh38)
    if 'Assembly' in chunk.columns:
        chunk = chunk[chunk['Assembly'] == 'GRCh38']

    if chunk.empty:
        continue

    clean_chunk = chunk[[col for col in cols_to_keep if col in chunk.columns]].copy()
    clean_chunk = clean_chunk.dropna(subset=['ClinicalSignificance', 'ReferenceAlleleVCF', 'AlternateAlleleVCF'])

    # clean_chunk['Label'] = clean_chunk['ClinicalSignificance'].astype(str).str.lower().apply(
    #     lambda x: 1 if 'pathogenic' in x else (0 if 'benign' in x else -1)
    # )
    # clean_chunk = clean_chunk[clean_chunk['Label'] != -1]

    def classify_significance(sig):
        s = str(sig).lower()
        if 'uncertain' in s or 'conflicting' in s or 'vus' in s:
            return 2
        if 'pathogenic' in s:
            return 1
        if 'benign' in s:
            return 0
        return -1

    clean_chunk['Label'] = clean_chunk['ClinicalSignificance'].apply(classify_significance)
    clean_chunk = clean_chunk[clean_chunk['Label'] != -1]

    for base in ['A', 'C', 'G', 'T']:
        clean_chunk[f'Ref_{base}'] = (clean_chunk['ReferenceAlleleVCF'] == base).astype(float)
        clean_chunk[f'Alt_{base}'] = (clean_chunk['AlternateAlleleVCF'] == base).astype(float)

    clean_chunk['Tab_Scaled_Submitters'] = np.log1p(pd.to_numeric(clean_chunk['NumberSubmitters'], errors='coerce').fillna(0))

    clean_chunk['Tab_Origin_Numeric'] = clean_chunk['OriginSimple'].astype(str).str.lower().apply(
        lambda x: 1.0 if 'germline' in x else (2.0 if 'somatic' in x else 0.0)
    )

    def map_review_to_stars(status_str):
        status = str(status_str).lower()
        if 'four stars' in status or 'practice guideline' in status: return 4.0
        if 'three stars' in status or 'reviewed by expert panel' in status: return 3.0
        if 'two stars' in status or 'criteria provided, multiple submitters' in status: return 2.0
        if 'one star' in status or 'criteria provided, single submitter' in status: return 1.0
        return 0.0

    clean_chunk['Tab_Review_Score'] = clean_chunk['ReviewStatus'].apply(map_review_to_stars)
    clean_chunk['Tab_Somatic_Impact'] = clean_chunk['SomaticClinicalImpact'].notna().astype(float)

    gene_counts = clean_chunk.groupby('GeneSymbol')['Label'].transform('count')
    gene_sums = clean_chunk.groupby('GeneSymbol')['Label'].transform('sum')
    clean_chunk['Tab_Gene_Vulnerability'] = (gene_sums + 1) / (gene_counts + 2)

    if clean_chunk.empty:
        continue

    clean_chunk.to_parquet(output_filename, index=False)
    print(f"-> Successfully processed Part {index}")

print("\nAll done!")

Processing started...
-> Successfully processed Part 0
-> Successfully processed Part 1
-> Successfully processed Part 2
-> Successfully processed Part 3
-> Successfully processed Part 4
-> Successfully processed Part 5
-> Successfully processed Part 6
-> Successfully processed Part 7
-> Successfully processed Part 8
-> Successfully processed Part 9
-> Successfully processed Part 10
-> Successfully processed Part 11
-> Successfully processed Part 12
-> Successfully processed Part 13
-> Successfully processed Part 14
-> Successfully processed Part 15
-> Successfully processed Part 16
-> Successfully processed Part 17
-> Successfully processed Part 18
-> Successfully processed Part 19
-> Successfully processed Part 20
-> Successfully processed Part 21
-> Successfully processed Part 22
-> Successfully processed Part 23
-> Successfully processed Part 24
-> Successfully processed Part 25
-> Successfully processed Part 26
-> Successfully processed Part 27
-> Successfully processed Part 28
->

In [ ]:
sample = pd.read_parquet('data/processed/clinvar_clean_part_23.parquet')
# sample.head(10)
sample.columns
# len(sample)

Index(['Type', 'GeneSymbol', 'ClinicalSignificance', 'Assembly', 'Chromosome',
       'Start', 'Stop', 'ReferenceAlleleVCF', 'AlternateAlleleVCF',
       'NumberSubmitters', 'OriginSimple', 'ReviewStatus',
       'SomaticClinicalImpact', 'Label', 'Ref_A', 'Alt_A', 'Ref_C', 'Alt_C',
       'Ref_G', 'Alt_G', 'Ref_T', 'Alt_T', 'Tab_Scaled_Submitters',
       'Tab_Origin_Numeric', 'Tab_Review_Score', 'Tab_Somatic_Impact',
       'Tab_Gene_Vulnerability'],
      dtype='object')

In [ ]:
import glob

processed_files = sorted(glob.glob('data/processed/clinvar_clean_part_*.parquet'))
print(len(processed_files))
all_sigs = set()
for fp in processed_files:
    df = pd.read_parquet(fp, columns=['ClinicalSignificance'])
    all_sigs.update(df['ClinicalSignificance'].unique())
print(sorted(all_sigs))

90
['Benign', 'Benign/Likely benign', 'Benign/Likely benign; association', 'Benign/Likely benign; drug response', 'Benign/Likely benign; drug response; other', 'Benign/Likely benign; other', 'Benign/Likely benign; other; risk factor', 'Benign/Likely benign; risk factor', 'Benign; Affects', 'Benign; Affects; association; other', 'Benign; association', 'Benign; confers sensitivity', 'Benign; drug response', 'Benign; other', 'Benign; protective', 'Benign; risk factor', 'Conflicting classifications of pathogenicity', 'Conflicting classifications of pathogenicity; Affects', 'Conflicting classifications of pathogenicity; association', 'Conflicting classifications of pathogenicity; association; risk factor', 'Conflicting classifications of pathogenicity; drug response', 'Conflicting classifications of pathogenicity; drug response; other', 'Conflicting classifications of pathogenicity; other', 'Conflicting classifications of pathogenicity; other; risk factor', 'Conflicting classifications of p

In [ ]:
print(master_df['Label'].value_counts())
print(master_df['Label'].value_counts(normalize=True) * 100)

Label
2    2409715
0    1290406
1     187517
Name: count, dtype: int64
Label
2    61.984038
0    33.192545
1     4.823417
Name: proportion, dtype: float64


### Standardizing Data
- Generate feature vectors wrt Reference and Alternate Alleles
- Map labels to ClinicalSignificance

In [ ]:
import pandas as pd
import glob
from tqdm.notebook import tqdm

processed_files = sorted(glob.glob('data/processed/clinvar_clean_part_*.parquet'))

print(len(processed_files))
# Label mapping
label_mapping = {
    # Pathogenic -> Class 1
    'Pathogenic': 1,
    'Pathogenic/Likely pathogenic': 1,
    'Likely pathogenic': 1,

    # Benign -> Class 0
    'Benign': 0,
    'Benign/Likely benign': 0,
    'Likely benign': 0,

    # VUS / Uncertain / Conflicting -> Class 2
    'Uncertain significance': 2,
    'Conflicting interpretations of pathogenicity': 2,
    'Uncertain significance, risk factor': 2,
    'Conflicting interpretations of pathogenicity, risk factor': 2
}

all_chunks = []

for file_path in tqdm(processed_files, desc="Processing Chunks with VUS"):
    df = pd.read_parquet(file_path)

    # Map raw clinical significance to 0, 1, or 2
    df['Label'] = df['ClinicalSignificance'].map(label_mapping)
    df = df.dropna(subset=['Label'])
    df['Label'] = df['Label'].astype(int)

    df['ReferenceAlleleVCF'] = df['ReferenceAlleleVCF'].str.upper()
    df['AlternateAlleleVCF'] = df['AlternateAlleleVCF'].str.upper()
    df = df[(df['ReferenceAlleleVCF'].str.len() == 1) & (df['AlternateAlleleVCF'].str.len() == 1)]

    all_chunks.append(df)

master_df = pd.concat(all_chunks, axis=0, ignore_index=True)

# Filter standard bases
standard_bases = ['A', 'T', 'G', 'C']
master_df = master_df[
    master_df['ReferenceAlleleVCF'].isin(standard_bases) &
    master_df['AlternateAlleleVCF'].isin(standard_bases)
]

columns_to_keep = columns_to_keep = [
    'Type', 'GeneSymbol', 'ClinicalSignificance', 'Label', 'Assembly', 'Chromosome',
    'Start', 'Stop', 'ReferenceAlleleVCF', 'AlternateAlleleVCF',
    'Ref_A', 'Ref_C', 'Ref_G', 'Ref_T',
    'Alt_A', 'Alt_C', 'Alt_G', 'Alt_T',
    'Tab_Scaled_Submitters', 'Tab_Origin_Numeric', 'Tab_Review_Score',
    'Tab_Somatic_Impact', 'Tab_Gene_Vulnerability'
]

master_df = master_df[columns_to_keep]
master_df.to_parquet('data/processed/clinvar_final_training_set.parquet', index=False)
print(f"Done! Total variants saved: {len(master_df)}")

90


Processing Chunks with VUS:   0%|          | 0/90 [00:00<?, ?it/s]

Done! Total variants saved: 3729663


### Sampling


In [ ]:
sample = pd.read_parquet('data/processed/clinvar_final_training_set.parquet')
sample.head()

,Type,GeneSymbol,ClinicalSignificance,Label,Assembly,Chromosome,Start,Stop,ReferenceAlleleVCF,AlternateAlleleVCF,...,Ref_T,Alt_A,Alt_C,Alt_G,Alt_T,Tab_Scaled_Submitters,Tab_Origin_Numeric,Tab_Review_Score,Tab_Somatic_Impact,Tab_Gene_Vulnerability
0,single nucleotide variant,ZNF592,Uncertain significance,2,GRCh38,15,84799209,84799209,G,A,...,0.0,1.0,0.0,0.0,0.0,0.693147,1.0,0.0,1.0,1.000000
1,single nucleotide variant,FOXRED1,Pathogenic,1,GRCh38,11,126275389,126275389,C,T,...,0.0,0.0,0.0,0.0,1.0,1.945910,1.0,2.0,1.0,0.800000
2,single nucleotide variant,FOXRED1,Likely pathogenic,1,GRCh38,11,126277517,126277517,A,G,...,0.0,0.0,0.0,1.0,0.0,1.098612,1.0,1.0,1.0,0.800000
3,single nucleotide variant,HFE,Uncertain significance,2,GRCh38,6,26091078,26091078,T,C,...,1.0,0.0,1.0,0.0,0.0,1.098612,1.0,1.0,1.0,1.428571
4,single nucleotide variant,HFE,Uncertain significance,2,GRCh38,6,26091041,26091041,G,C,...,0.0,0.0,1.0,0.0,0.0,1.098612,1.0,1.0,1.0,1.428571


##  Load processed data


In [ ]:
import os
import pandas as pd
from google.colab import drive
drive.mount('/content/drive')
loc = '/content/drive/MyDrive/variant_effect_prediction/data/processed/clinvar_final_training_set.parquet'
master_df = pd.read_parquet(loc)
master_df.head()

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


,Type,GeneSymbol,ClinicalSignificance,Label,Assembly,Chromosome,Start,Stop,ReferenceAlleleVCF,AlternateAlleleVCF,...,Ref_T,Alt_A,Alt_C,Alt_G,Alt_T,Tab_Scaled_Submitters,Tab_Origin_Numeric,Tab_Review_Score,Tab_Somatic_Impact,Tab_Gene_Vulnerability
0,single nucleotide variant,ZNF592,Uncertain significance,2,GRCh38,15,84799209,84799209,G,A,...,0.0,1.0,0.0,0.0,0.0,0.693147,1.0,0.0,1.0,1.000000
1,single nucleotide variant,FOXRED1,Pathogenic,1,GRCh38,11,126275389,126275389,C,T,...,0.0,0.0,0.0,0.0,1.0,1.945910,1.0,2.0,1.0,0.800000
2,single nucleotide variant,FOXRED1,Likely pathogenic,1,GRCh38,11,126277517,126277517,A,G,...,0.0,0.0,0.0,1.0,0.0,1.098612,1.0,1.0,1.0,0.800000
3,single nucleotide variant,HFE,Uncertain significance,2,GRCh38,6,26091078,26091078,T,C,...,1.0,0.0,1.0,0.0,0.0,1.098612,1.0,1.0,1.0,1.428571
4,single nucleotide variant,HFE,Uncertain significance,2,GRCh38,6,26091041,26091041,G,C,...,0.0,0.0,1.0,0.0,0.0,1.098612,1.0,1.0,1.0,1.428571


In [ ]:
master_df.columns

Index(['Type', 'GeneSymbol', 'ClinicalSignificance', 'Label', 'Assembly',
       'Chromosome', 'Start', 'Stop', 'ReferenceAlleleVCF',
       'AlternateAlleleVCF', 'Ref_A', 'Ref_C', 'Ref_G', 'Ref_T', 'Alt_A',
       'Alt_C', 'Alt_G', 'Alt_T', 'Tab_Scaled_Submitters',
       'Tab_Origin_Numeric', 'Tab_Review_Score', 'Tab_Somatic_Impact',
       'Tab_Gene_Vulnerability'],
      dtype='object')

In [ ]:
import torch
import pandas as pd

class_counts = master_df['Label'].value_counts().sort_index()
class_names = {0: "Benign", 1: "Pathogenic", 2: "VUS"}

print("--- Class Distribution ---")
for label_idx, count in class_counts.items():
    print(f"  Class {label_idx} ({class_names[label_idx]}): {count:,} ({count / len(master_df):.2%})")

# Calculate inverse-frequency class weights for CrossEntropyLoss
total_samples = len(master_df)
num_classes = 3
class_weights = total_samples / (num_classes * class_counts.values)
class_weights_tensor = torch.tensor(class_weights, dtype=torch.float32).to(
    torch.device("cuda" if torch.cuda.is_available() else "cpu")
)

print(f"\nCalculated Class Weights: {class_weights_tensor}")

--- Class Distribution ---
  Class 0 (Benign): 1,290,313 (34.60%)
  Class 1 (Pathogenic): 187,184 (5.02%)
  Class 2 (VUS): 2,252,166 (60.39%)

Calculated Class Weights: tensor([0.9635, 6.6417, 0.5520])


# Step 1: PyTorch Data Loading

## Leakage free data split

### Process Split

In [ ]:
import os
os.getcwd()

'/content/drive/MyDrive/variant_effect_prediction'

In [ ]:

from sklearn.model_selection import GroupShuffleSplit

gss = GroupShuffleSplit(n_splits=1, train_size=0.8, random_state=42)
train_ind, test_ind = next(gss.split(master_df, groups=master_df['GeneSymbol']))

train_df = master_df.iloc[train_ind]
test_df = master_df.iloc[test_ind]

loc = '/content/drive/MyDrive/variant_effect_prediction/data/processed/'

train_df.to_parquet(f"{loc}train_data.parquet", index=False)
test_df.to_parquet(f"{loc}test_data.parquet", index=False)

print("Done!")

len(train_df), len(test_df), len(master_df), len(master_df) * 0.8

Done!


(3009291, 720372, 3729663, 2983730.4000000004)

### Load Split Data

In [ ]:
import pandas as pd

loc = '/content/drive/MyDrive/variant_effect_prediction/data/processed/'
train_df = pd.read_parquet(f"{loc}train_data.parquet")
test_df = pd.read_parquet(f"{loc}test_data.parquet")
len(train_df), train_df.columns

(3009291,
 Index(['Type', 'GeneSymbol', 'ClinicalSignificance', 'Label', 'Assembly',
        'Chromosome', 'Start', 'Stop', 'ReferenceAlleleVCF',
        'AlternateAlleleVCF', 'Ref_A', 'Ref_C', 'Ref_G', 'Ref_T', 'Alt_A',
        'Alt_C', 'Alt_G', 'Alt_T', 'Tab_Scaled_Submitters',
        'Tab_Origin_Numeric', 'Tab_Review_Score', 'Tab_Somatic_Impact',
        'Tab_Gene_Vulnerability'],
       dtype='object'))

## Type Conversion and Batch Loader

In [ ]:
import torch
from torch.utils.data import Dataset, DataLoader

class VariantDataset(Dataset):
  def __init__(self, dataframe):
    feature_cols = [
        col for col in dataframe.columns
        if col.startswith('Ref_') or col.startswith('Alt_')
    ]

    self.X = torch.tensor(dataframe[feature_cols].values, dtype=torch.float32)
    self.y = torch.tensor(dataframe['Label'].values, dtype=torch.float32)

  def __len__(self):
    return len(self.y)

  def __getitem__(self, idx):
    return self.X[idx], self.y[idx]

train_data = VariantDataset(train_df)
test_data = VariantDataset(test_df)

train_loader = DataLoader(train_data, batch_size=128, shuffle=True)
test_loader = DataLoader(test_data, batch_size=128, shuffle=False)

first_batch_features, first_batch_labels = next(iter(train_loader))

print("--- PyTorch Infrastructure Verification ---")
print(f"Batch Features Matrix Shape: {first_batch_features.shape}")
print(f"Batch Labels Vector Shape:   {first_batch_labels.shape}")

--- PyTorch Infrastructure Verification ---
Batch Features Matrix Shape: torch.Size([128, 8])
Batch Labels Vector Shape:   torch.Size([128])


In [ ]:
first_batch_features.shape, first_batch_features.view(-1, 4, 2).shape

(torch.Size([64, 8]), torch.Size([64, 4, 2]))

In [ ]:
first_batch_features.reshape(-1, 4, 2)[0], first_batch_features[0]

(tensor([[1., 0.],
         [0., 0.],
         [0., 0.],
         [1., 0.]]),
 tensor([1., 0., 0., 0., 0., 0., 1., 0.]))

# Step 2: Build Model Architecture

## Arm 1: The Multi-Armed Network Architecture

In [ ]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
device

device(type='cuda')

In [ ]:
import torch
import pandas as pd
from torch.utils.data import Dataset, DataLoader

class MultiArmedDataset(Dataset):
    def __init__(self, dataframe):
        # 8 Sequence Arm Switches
        seq_cols = ['Ref_A', 'Alt_A', 'Ref_C', 'Alt_C', 'Ref_G', 'Alt_G', 'Ref_T', 'Alt_T']
        self.X_seq = torch.tensor(dataframe[seq_cols].values, dtype=torch.float32)

        # 5 Tabular Arm Biomarkers
        tab_cols = [
            'Tab_Scaled_Submitters',
            'Tab_Origin_Numeric',
            'Tab_Review_Score',
            'Tab_Somatic_Impact',
            'Tab_Gene_Vulnerability'
        ]
        self.X_tab = torch.tensor(dataframe[tab_cols].values, dtype=torch.float32)

        labels = dataframe['Label'].values
        labels = np.clip(labels, 0.0, 1.0)
        self.y = torch.tensor(labels, dtype=torch.float32)

    def __len__(self):
        return len(self.y)

    def __getitem__(self, idx):
        return self.X_seq[idx], self.X_tab[idx], self.y[idx]

train_data = MultiArmedDataset(train_df)

test_data = MultiArmedDataset(test_df)

train_loader = DataLoader(train_data, batch_size=64, shuffle=True)
test_loader = DataLoader(test_data, batch_size=64, shuffle=False)

seq1, tab1, y1 = next(iter(train_loader))
num_tab_features = tab1.shape[1]

print(f"Total feature columns: {num_tab_features}")
seq1.shape, tab1.shape

Total feature columns: 5


(torch.Size([64, 8]), torch.Size([64, 5]))

In [ ]:
import torch
import torch.nn as nn

class ProductionMultiArmedPredictor(nn.Module):
    def __init__(self, tabular_features_dim=5):
        super(ProductionMultiArmedPredictor, self).__init__()

        # --- ARM 1: CNN Sequence Branch ---
        # in_channels=4 covers [A, C, G, T]
        # kernel_size=2 slides across [Ref, Alt]
        self.seq_net = nn.Sequential(
            nn.Conv1d(in_channels=4, out_channels=32, kernel_size=2, padding=0),
            nn.ReLU(),
            nn.Flatten() # Flattens to [batch_size, 32]
        )

        # --- ARM 2: Dense MLP Tabular Branch ---
        self.tab_net = nn.Sequential(
            nn.Linear(tabular_features_dim, 16),
            nn.ReLU() # Outputs [batch_size, 16]
        )

        # --- LATE FUSION HEAD ---
        # 32 (CNN) + 16 (MLP) = 48 fused features
        self.fc = nn.Sequential(
            nn.Linear(32 + 16, 16),
            nn.ReLU(),
            nn.Linear(16, 1),
            nn.Sigmoid()
        )

    def forward(self, x_seq, x_tab):
        x_seq = x_seq.reshape(-1, 4, 2)
        x_seq = self.seq_net(x_seq)
        x_tab = self.tab_net(x_tab)
        x_combined = torch.cat((x_seq, x_tab), dim=1)
        return self.fc(x_combined)

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
model = ProductionMultiArmedPredictor(tabular_features_dim=5).to(device)
model

ProductionMultiArmedPredictor(
  (seq_net): Sequential(
    (0): Conv1d(4, 32, kernel_size=(2,), stride=(1,))
    (1): ReLU()
    (2): Flatten(start_dim=1, end_dim=-1)
  )
  (tab_net): Sequential(
    (0): Linear(in_features=5, out_features=16, bias=True)
    (1): ReLU()
  )
  (fc): Sequential(
    (0): Linear(in_features=48, out_features=16, bias=True)
    (1): ReLU()
    (2): Linear(in_features=16, out_features=1, bias=True)
    (3): Sigmoid()
  )
)

In [ ]:
import torch.optim as optim
from tqdm.notebook import tqdm

weight_tensor = torch.tensor([10.0]).to(device)
loss_fn = nn.BCELoss(weight=weight_tensor)
optimizer = optim.Adam(model.parameters(), lr=0.001)

pos_weight = 6.3
EPOCHS = 5
print("Starting Training")

for epoch in range(EPOCHS):
    model.train()
    train_loss = 0.0

    loop = tqdm(train_loader, desc=f'Epoch {epoch + 1}/{EPOCHS}')
    for seq, tab, targets in loop:
        targets = targets.float()
        seq, tab, targets = seq.to(device), tab.to(device), targets.unsqueeze(-1).to(device)

        optimizer.zero_grad()
        preds_batch = model(seq, tab)
        loss = loss_fn(preds_batch, targets)

        loss.backward()
        optimizer.step()

        train_loss += loss.item() * seq.size(0)

    epoch_loss = train_loss / len(train_loader.dataset)
    print(f"Epoch {epoch+1} Completed -> Weighted Loss: {epoch_loss:.4f}\n")

print("Training complete!")

Starting Training


Epoch 1/5:   0%|          | 0/47021 [00:00<?, ?it/s]

Epoch 1 Completed -> Weighted Loss: 3.6102



Epoch 2/5:   0%|          | 0/47021 [00:00<?, ?it/s]

Epoch 2 Completed -> Weighted Loss: 3.6008



...

Epoch 3 Completed -> Weighted Loss: 3.5965



Epoch 4/5:   0%|          | 0/47021 [00:00<?, ?it/s]

Epoch 4 Completed -> Weighted Loss: 3.5910



Epoch 5/5:   0%|          | 0/47021 [00:00<?, ?it/s]

Epoch 5 Completed -> Weighted Loss: 3.5872

Training complete!


In [51]:
import numpy as np
from sklearn.metrics import (accuracy_score, precision_score, recall_score,
                              f1_score, precision_recall_curve, roc_auc_score)

model.eval()
all_probs = []
all_labels = []

with torch.no_grad():
    for seq, tab, targets in test_loader:
        seq, tab = seq.to(device), tab.to(device)
        pred_prob = model(seq, tab).squeeze(-1).cpu().numpy()
        all_probs.extend(pred_prob)
        all_labels.extend(targets.numpy())

all_probs  = np.array(all_probs)
all_labels = np.array(all_labels)

# Find threshold that maximises F1
precisions, recalls, thresholds = precision_recall_curve(all_labels, all_probs)
f1_scores  = 2 * precisions * recalls / (precisions + recalls + 1e-8)
best_idx   = np.argmax(f1_scores)
best_thr   = thresholds[best_idx]
print(f"Optimal threshold: {best_thr:.4f}  (F1={f1_scores[best_idx]:.4f})")

preds = (all_probs >= best_thr).astype(int)

acc  = accuracy_score(all_labels, preds)
prec = precision_score(all_labels, preds, zero_division=0)
rec  = recall_score(all_labels, preds, zero_division=0)
f1   = f1_score(all_labels, preds, zero_division=0)
auc  = roc_auc_score(all_labels, all_probs)

print("\n================ MULTI-ARMED NETWORK METRICS ================")
print(f"Test Accuracy:  {acc  * 100:.2f}%")
print(f"Precision:      {prec:.4f}")
print(f"Recall:         {rec :.4f}")
print(f"F1-Score:       {f1  :.4f}")
print(f"ROC-AUC:        {auc :.4f}")
print("=============================================================")

Optimal threshold: 0.4531  (F1=0.8882)

================ MULTI-ARMED NETWORK METRICS ================
Test Accuracy:  84.59%
Precision:      0.8396
Recall:         0.9429
F1-Score:       0.8882
ROC-AUC:        0.9093


In [53]:
import torch
weights_path = "/content/drive/MyDrive/variant_effect_prediction/data/processed/arm1:MultiArm.pth"
torch.save(model.state_dict(), weights_path)
print(f"Baseline weights successfully saved to Google Drive at:\n{weights_path}")
# model.load_state_dict(torch.load(weights_path))

Baseline weights successfully saved to Google Drive at:
/content/drive/MyDrive/variant_effect_prediction/data/processed/arm1:MultiArm.pth


In [54]:
import torch
import json

# 1. Save PyTorch Model Weights
torch.save(model.state_dict(), 'multi_armed_model_84acc.pth')

# 2. Save feature configuration for the API preprocessor
config = {
    'seq_cols': ['Ref_A', 'Alt_A', 'Ref_C', 'Alt_C', 'Ref_G', 'Alt_G', 'Ref_T', 'Alt_T'],
    'tab_cols': [
        'Tab_Scaled_Submitters',
        'Tab_Origin_Numeric',
        'Tab_Review_Score',
        'Tab_Somatic_Impact',
        'Tab_Gene_Vulnerability'
    ]
}

with open('model_config.json', 'w') as f:
    json.dump(config, f)

print("Model weights and configs saved successfully!")

Model weights and configs saved successfully!
